In [34]:
import sys

In [35]:
%%capture
try:
    # Attempt to import a module that's only available in Colab
    from google.colab import drive
    in_colab = True
except ImportError:
    in_colab = False

if in_colab:
    # Colab specific setup
    drive.mount('/content/drive')
    sys.path.append('/content/drive/MyDrive/structure-loss-classification/')
    my_local_data = '/content/drive/MyDrive/types/'
    %cd '/content/drive/MyDrive/structure-loss-classification/'
    %pip install scienceplots
    %pip install pytorch_lightning
else:
    # Local machine setup
    my_local_data = "/mnt/g/My Drive/types/"

In [28]:
import torch
import torch.nn as nn

import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader, SubsetRandomSampler
from torchvision.models.feature_extraction import (
    get_graph_node_names,
    create_feature_extractor,
)

In [40]:
from sklearn.model_selection import train_test_split, StratifiedKFold

In [30]:
import pytorch_lightning as pl

In [31]:
import numpy as np
import matplotlib.pyplot as plt
import scienceplots

plt.style.use(["science", "notebook", "grid"])

In [32]:
import pickle

In [33]:
from models.models import LeNet5
from lightning_modules.lightning_modules import LitLeNet5
from visualization.filters import display_filters
from datasets.data_modules import CustomImageDataModule
from train.train import get_features, train_model

In [36]:
ToTensorAndNormalize = transforms.Compose(
    [
        transforms.Resize((244, 244)),
        # transforms.RandomHorizontalFlip(),
        # transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),  # mean  # std
    ]
)
aux_data = datasets.ImageFolder(root=my_local_data,
                                transform=ToTensorAndNormalize)

In [37]:
# Try to load cached targets first
try:
    with open(f"logdir/cached_targets.pkl", "rb") as f:
        targets = pickle.load(f)
except FileNotFoundError:
    targets = [t for _, t in aux_data]
    # Cache the targets for next time
    with open(f"logdir/cached_targets.pkl", "wb") as f:
        pickle.dump(targets, f)

In [38]:
model2 = LitLeNet5(num_classes=4, learning_rate=0.001)

In [39]:
dataset = datasets.ImageFolder(my_local_data)
print(f"Number of classes: {len(dataset.classes)}")
print(f"Class to index mapping: {dataset.class_to_idx}")

Number of classes: 4
Class to index mapping: {'goodIngots': 0, 'typeA': 1, 'typeB': 2, 'typeC': 3}


In [15]:
kfold = StratifiedKFold(
    n_splits=4,
    shuffle=True,
)

In [16]:
validation_metrics = []

In [17]:
trainer_config = {
    "patience": 3,
    "accelerator": "gpu",
    "devices": -1,
    "max_epochs": 10,
    "precision": 32,
    "n_steps": 5,
}

In [18]:
# Assuming aux_data is a dataset object and targets are the labels
train_idx, val_idx, _, _ = train_test_split(
    range(len(aux_data)), targets, test_size=0.2, random_state=42
)

train_data = torch.utils.data.Subset(aux_data, train_idx)
val_data = torch.utils.data.Subset(aux_data, val_idx)

In [19]:
data_module = CustomImageDataModule(
    train_dataset=train_data,
    val_dataset=val_data,
    batch_size=16,
    num_workers=4,
)

In [ ]:
# Assuming model2 is initialized, and trainer_config is defined
val_metrics = train_model(
    model=model2,
    trainer_config=trainer_config,
    save_dir="logdir/",
    data_module=data_module,
)